<a href="https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked Content Actions

The purpose of this queue is to rank content for human review using the model's estimated decline probability.

The action thresholds are:

- **Review first**: decline probability >= 0.80
- **Prioritize review**: decline probability >= 0.60 and < 0.80
- **Monitor**: decline probability >= 0.40 and < 0.60
- **Protect**: decline probability < 0.40

### Reason codes

- **HIGH_DECLINE_RISK**: high estimated probability of decline
- **MEDIUM_DECLINE_RISK**: moderate estimated probability of decline
- **LOW_DECLINE_RISK**: lower estimated probability of decline
- **PROTECT**: no strong model signal for decline

The queue is a decision-support tool. A high score means that the content should be reviewed first; it does not mean that the content must be changed.

The model score is an observed and measured ranking signal from this dataset. It should not be interpreted as proof that a content change will improve search performance.

In [6]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(
    "https://raw.githubusercontent.com/shivanilokh/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)

# ==========================================
# CREATE TARGET
# ==========================================

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

target = "is_declining_label"

# ==========================================
# REMOVE TARGET / ID COLUMNS
# ==========================================

exclude_cols = {
    target,
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
}

feature_cols = [
    c for c in df.columns
    if c not in exclude_cols
]

print("Number of model features:", len(feature_cols))

# ==========================================
# BUILD MODEL
# ==========================================

def build_model(X_train, y_train):

    numeric_features = X_train.select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ]
    )

    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", rf)
        ]
    )

    model.fit(X_train, y_train)

    return model


# ==========================================
# CLIENT-GROUPED SPLIT
# ==========================================

clients = df["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_grouped = df[
    df["client_id"].isin(train_clients)
].copy()

test_grouped = df[
    df["client_id"].isin(test_clients)
].copy()

print("Training rows:", len(train_grouped))
print("Test rows:", len(test_grouped))

# ==========================================
# TRAIN MODEL
# ==========================================

grouped_model = build_model(
    train_grouped[feature_cols],
    train_grouped[target]
)

# ==========================================
# CREATE RANKED QUEUE
# ==========================================

queue = test_grouped[
    [
        "content_id",
        "client_id",
        "freshness_tier",
        "age_tier",
        "days_since_last_update",
        "content_age_days"
    ]
].copy()

queue["decline_probability"] = grouped_model.predict_proba(
    test_grouped[feature_cols]
)[:, 1]

# ==========================================
# ACTION
# ==========================================

def assign_action(probability):

    if probability >= 0.80:
        return "Review first"

    elif probability >= 0.60:
        return "Prioritize review"

    elif probability >= 0.40:
        return "Monitor"

    else:
        return "Protect"


def assign_reason(probability):

    if probability >= 0.80:
        return "HIGH_DECLINE_RISK"

    elif probability >= 0.60:
        return "MEDIUM_DECLINE_RISK"

    elif probability >= 0.40:
        return "LOW_DECLINE_RISK"

    else:
        return "PROTECT"


queue["action"] = queue["decline_probability"].apply(
    assign_action
)

queue["reason_code"] = queue["decline_probability"].apply(
    assign_reason
)

# ==========================================
# ARCHETYPE
# ==========================================

def assign_archetype(row):

    freshness = str(row["freshness_tier"]).lower()
    age = str(row["age_tier"]).lower()

    if "stale" in freshness or "old" in age:
        return "Older or stale content"

    elif "fresh" in freshness or "new" in age:
        return "Fresh or newer content"

    else:
        return "Other content"


queue["archetype"] = queue.apply(
    assign_archetype,
    axis=1
)

# ==========================================
# SORT QUEUE
# ==========================================

queue = queue.sort_values(
    "decline_probability",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# ==========================================
# SHOW TOP QUEUE
# ==========================================

print("\nTop 20 ranked actions:")

display(
    queue[
        [
            "rank",
            "content_id",
            "decline_probability",
            "action",
            "reason_code",
            "archetype"
        ]
    ].head(20)
)

print("\nAction counts:")
display(queue["action"].value_counts())

print("\nReason-code counts:")
display(queue["reason_code"].value_counts())

Dataset shape: (30000, 44)
Number of model features: 40
Training rows: 26581
Test rows: 3419

Top 20 ranked actions:


,rank,content_id,decline_probability,action,reason_code,archetype
0,1,content_c92cbdb448d0,0.983333,Review first,HIGH_DECLINE_RISK,Other content
1,2,content_6752b50ee577,0.976667,Review first,HIGH_DECLINE_RISK,Other content
2,3,content_ca122cc888aa,0.976667,Review first,HIGH_DECLINE_RISK,Other content
3,4,content_7ba9813c1beb,0.973333,Review first,HIGH_DECLINE_RISK,Other content
4,5,content_78d59b5950e9,0.973333,Review first,HIGH_DECLINE_RISK,Other content
5,6,content_870430512868,0.973333,Review first,HIGH_DECLINE_RISK,Other content
6,7,content_b73061588e7d,0.973333,Review first,HIGH_DECLINE_RISK,Other content
7,8,content_5ad0d416fbbd,0.970000,Review first,HIGH_DECLINE_RISK,Other content
8,9,content_29884c0f9255,0.970000,Review first,HIGH_DECLINE_RISK,Other content
9,10,content_9de926fa1505,0.970000,Review first,HIGH_DECLINE_RISK,Other content



Action counts:


,count
action,
Prioritize review,997
Protect,988
Review first,800
Monitor,634



Reason-code counts:


,count
reason_code,
MEDIUM_DECLINE_RISK,997
PROTECT,988
HIGH_DECLINE_RISK,800
LOW_DECLINE_RISK,634


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended for content analysts or SEO reviewers who need to decide which content should be checked first.

The model provides a ranked review queue based on an estimated probability of decline. The queue can help a reviewer spend limited time on higher-priority items first.

The intended use is decision support, not automatic content management.

### Limits

The model was evaluated on this anonymized dataset using a client-grouped validation design. The measured Precision@50 was 1.00 in the Week-6 validation run, with zero client overlap between training and test groups.

This result is observed on this dataset and should not be treated as a guarantee of future performance.

The label is derived from the observed trend direction, so the model identifies patterns associated with declining content. It does not prove that a refresh or other content action will cause an improvement.

The queue should therefore be reviewed by a human before any content change is made.

In [7]:
print("INTENDED USE CHECK")
print("------------------")
print("Purpose: rank content for human review.")
print("Decision type: decision-support.")
print("Automatic content changes: not allowed.")

print("\nValidation receipt:")
print("Validation design: client-grouped split")
print("Client overlap:", len(
    set(train_grouped["client_id"]) &
    set(test_grouped["client_id"])
))

print("\nQueue rows:", len(queue))
print("Top-ranked probability:",
      round(queue["decline_probability"].max(), 3))

INTENDED USE CHECK
------------------
Purpose: rank content for human review.
Decision type: decision-support.
Automatic content changes: not allowed.

Validation receipt:
Validation design: client-grouped split
Client overlap: 0

Queue rows: 3419
Top-ranked probability: 0.983


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Rules

A person must review the content before taking action.

For items marked **Review first**, the reviewer should check:

1. Whether the observed decline signal is recent and meaningful.
2. Whether the page is still relevant to the intended search intent.
3. Whether the content is accurate, useful, and complete.
4. Whether the page has already been updated recently.
5. Whether there is a valid business or user reason to change the page.
6. Whether the recommended action has a reasonable expected value compared with the effort required.

### No-go list

The model should never automatically:

- publish or edit content,
- delete a page,
- change search strategy,
- make a business decision,
- guarantee ranking improvement,
- treat model probability as proof of causation,
- override an expert's review,
- make changes to content without checking the page itself.

The model only prioritizes review. The final decision remains with a human reviewer.

In [8]:
# Simple human-review summary

review_summary = pd.DataFrame({
    "Action": [
        "Review first",
        "Prioritize review",
        "Monitor",
        "Protect"
    ],
    "Meaning": [
        "Highest review priority",
        "Review after highest-priority items",
        "Keep under observation",
        "No strong decline signal"
    ],
    "Human review required": [
        "Yes",
        "Yes",
        "Recommended",
        "Optional"
    ]
})

display(review_summary)

print("No automatic content changes are performed by this notebook.")

,Action,Meaning,Human review required
0,Review first,Highest review priority,Yes
1,Prioritize review,Review after highest-priority items,Yes
2,Monitor,Keep under observation,Recommended
3,Protect,No strong decline signal,Optional


No automatic content changes are performed by this notebook.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring Plan

The playbook should be checked periodically because content performance and search behavior can change.

The following signals would indicate that the recommendations may be becoming stale:

- Precision@50 falls meaningfully below the validated 1.00 result on a comparable evaluation sample.
- The distribution of decline probabilities changes substantially.
- The proportion of content receiving each action changes unexpectedly.
- Freshness or content-age patterns change compared with the data used to build the model.
- New content types or data fields appear that were not represented during training.
- The relationship between the model score and the observed decline outcome weakens.

### Retrain trigger

Retraining should be considered when new labeled data is available and the model shows sustained degradation on a fresh validation sample.

A new model should not be deployed only because a scheduled date has arrived. It should first pass the same leakage checks and client-grouped validation approach.

### Practical review cadence

A lightweight monitoring review can be performed monthly or whenever a meaningful data or content-system change occurs.

The model is a research prototype, so monitoring is intended to identify when the analysis should be revisited rather than to operate as a production alerting system.

In [9]:
# Monitoring checks for the current queue

print("MONITORING SNAPSHOT")
print("-------------------")

print("Queue size:", len(queue))

print("\nAction distribution:")
action_distribution = (
    queue["action"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(action_distribution.rename("percent"))

print("\nDecline probability summary:")
display(
    queue["decline_probability"].describe()
)

print("\nArchetype distribution:")
display(
    queue["archetype"].value_counts()
)

print("\nRetrain should be considered if future validation shows sustained performance degradation.")

MONITORING SNAPSHOT
-------------------
Queue size: 3419

Action distribution:


,percent
action,
Prioritize review,29.16
Protect,28.90
Review first,23.40
Monitor,18.54



Decline probability summary:


,decline_probability
count,3419.000000
mean,0.549770
std,0.282339
min,0.000000
25%,0.333333
50%,0.620000
75%,0.786667
max,0.983333



Archetype distribution:


,count
archetype,
Other content,3419



Retrain should be considered if future validation shows sustained performance degradation.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Paper Exports

The ranked queue is exported to `work/outputs/` so that the research paper can reuse the same recommendations without manually copying results.

The exported queue contains the ranking, model score, action, reason code, and content archetype.

The queue is a regenerated artifact and is not intended to be treated as a production database.

The export is generated by the notebook so the paper can trace its recommendation section back to the analysis.

In [10]:
import os
import json

# ==========================================
# CREATE OUTPUT DIRECTORY
# ==========================================

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

# ==========================================
# EXPORT RANKED QUEUE
# ==========================================

queue_export = queue[
    [
        "rank",
        "content_id",
        "decline_probability",
        "action",
        "reason_code",
        "archetype",
        "freshness_tier",
        "age_tier",
        "days_since_last_update",
        "content_age_days"
    ]
].copy()

queue_path = os.path.join(
    output_dir,
    "w07_ranked_action_queue.csv"
)

queue_export.to_csv(
    queue_path,
    index=False
)

# ==========================================
# EXPORT SUMMARY
# ==========================================

summary = {
    "dataset_rows": int(len(df)),
    "queue_rows": int(len(queue)),
    "validation_design": "client-grouped split",
    "client_overlap": int(
        len(
            set(train_grouped["client_id"]) &
            set(test_grouped["client_id"])
        )
    ),
    "action_thresholds": {
        "review_first": ">= 0.80",
        "prioritize_review": ">= 0.60 and < 0.80",
        "monitor": ">= 0.40 and < 0.60",
        "protect": "< 0.40"
    },
    "intended_use": "decision-support and human review prioritization",
    "automatic_content_changes": False
}

summary_path = os.path.join(
    output_dir,
    "w07_action_playbook_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("EXPORT COMPLETE")
print("----------------")
print("Queue:", queue_path)
print("Summary:", summary_path)

print("\nFiles created:")
print(os.listdir(output_dir))

EXPORT COMPLETE
----------------
Queue: work/outputs/w07_ranked_action_queue.csv
Summary: work/outputs/w07_action_playbook_summary.json

Files created:
['w07_ranked_action_queue.csv', 'w07_action_playbook_summary.json']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Final self-check

- [x] Ranked actions and reason codes are defined.
- [x] Intended use and limits are stated.
- [x] Human review rules and no-go cases are stated.
- [x] Monitoring and retrain triggers are defined.
- [x] The ranked queue is exported for the research paper.
- [x] The notebook uses decision-support language rather than production or causal claims.
- [x] The notebook was run from top to bottom without errors.